# 00: Environment Setup — Run Me Before Any Lesson!

## What This Notebook Does

- Installs all required Python packages **in one go**
- Helps you configure your API key securely
- Tests your connection to OpenAI, DeepSeek, OpenRouter and local Ollama

> Run this notebook **before Lesson 01**. After that, all other lessons will work without any extra setup.

## Prerequisites

- Python 3.8+ installed
- An API key from **one** of:
  - **OpenAI** — https://platform.openai.com/api-keys
  - **DeepSeek** — https://platform.deepseek.com/api_keys
  - **OpenRouter** — https://openrouter.ai/keys
  - or no key at all, using local **Ollama** (see Step 2.5)

## Step 1: Install All Dependencies

This installs every package needed across all 10 lessons. It only needs to run once per environment.

In [1]:
# Install all packages at once — this may take 1-2 minutes
!pip install -r requirements.txt -q
print("All packages installed successfully!")

All packages installed successfully!


## Step 2: Configure Your API Key

### Option A: Create a `.env` file (recommended)

Create a file named `.env` in the same folder as this notebook and add your key:

```
OPENAI_API_KEY=sk-your-actual-key-here
# OR
DEEPSEEK_API_KEY=sk-your-deepseek-key-here
# OR
OPENROUTER_API_KEY=sk-or-v1-your-actual-key-here
```

**You only need ONE of these keys.**

> This repo ships a `.gitignore` that already excludes `.env`, so your key will not be committed by accident.

### Option B: Enter your key directly below

If you prefer not to create a file, you can enter your key directly in this notebook.

In [2]:
import os
from pathlib import Path

# Check if .env file exists
env_path = Path('.env')
if env_path.exists():
    print("Found .env file. Loading API keys from it...")
    from dotenv import load_dotenv
    load_dotenv()
else:
    print("No .env file found. You can create one or enter your key below.")
    print("\nCopy .env.example to get started:")
    print("  cp .env.example .env")
    print("  Then edit .env and add your API key.")
    print("\nFor now, enter your key below (it won't be shown on screen):")
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
    print("Key set for this session. For permanent use, create a .env file.")

Found .env file. Loading API keys from it...


## Step 2.5: Set Up Ollama (Optional — Local & Free)

Ollama lets you run AI models **locally on your computer**, completely free and offline.

### Installation

1. Download Ollama from https://ollama.com/download (macOS / Linux / Windows)
2. Install and start the service:
   ```bash
   ollama serve
   ```
3. Pull a model (pick one):
   ```bash
   ollama pull gemma4:e2b-mlx    # Google Gemma 4, MLX build for Apple Silicon, ~6.5GB
   ollama pull qwen2.5     # Alibaba's Qwen 2.5, ~4GB
   ollama pull mistral     # Mistral 7B, ~4GB
   ```
4. Verify: `ollama list` should show your downloaded models

> `gemma4:e2b-mlx` is just a proof of concept — a small 2B model that shows the notebooks
> run with zero API spend. Pull anything from https://ollama.com/library that your machine
> can handle and put that name in the `ollama` block instead. Rule of thumb: a 4-bit model
> needs roughly 1GB of RAM per billion parameters, so 16GB comfortably runs a 7–8B model.
> Bigger local models give noticeably better results in the evaluation and judging lessons.

> Ollama is compatible with the OpenAI API format. We connect via `http://localhost:11434/v1`.
> No API key needed — just make sure `ollama serve` is running in the background.

## Step 3: Verify Your Connection

Test all three API options (OpenAI, OpenRouter, and Ollama) to make sure everything is working.

In [3]:
from openai import OpenAI
import os

# The four providers the lessons support. All of them speak the OpenAI API format.
PROVIDERS = {
    'OpenAI':     {'base_url': None,                            'env': 'OPENAI_API_KEY',     'model': 'gpt-5.6-luna'},
    'DeepSeek':   {'base_url': 'https://api.deepseek.com/v1',    'env': 'DEEPSEEK_API_KEY',   'model': 'deepseek-v4-flash'},
    'OpenRouter': {'base_url': 'https://openrouter.ai/api/v1',   'env': 'OPENROUTER_API_KEY', 'model': 'openai/gpt-5.6-luna'},
    # Ollama runs locally and needs no key — just `ollama serve` + `ollama pull gemma4:e2b-mlx`
    'Ollama':     {'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
                   'env': None, 'model': 'gemma4:e2b-mlx'},
}


def test_provider(label, cfg):
    """Send one tiny request to a provider and report the result."""
    api_key = 'ollama' if cfg['env'] is None else os.getenv(cfg['env'])
    if not api_key:
        print(f"[SKIP] {label}: no {cfg['env']} found in your environment / .env file.")
        return
    try:
        client = OpenAI(api_key=api_key, base_url=cfg['base_url'])
        r = client.chat.completions.create(
            model=cfg['model'],
            messages=[{'role': 'user', 'content': 'Say hello in one sentence.'}],
            max_tokens=100
        )
        print(f"[OK] {label} ({cfg['model']}): {r.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"[FAIL] {label}: {str(e)[:150]}")


for label, cfg in PROVIDERS.items():
    test_provider(label, cfg)

print("\nYou only need ONE of these to work.")


[SKIP] OpenAI: no OPENAI_API_KEY found in your environment / .env file.
[OK] DeepSeek (deepseek-v4-flash): Hello!
[SKIP] OpenRouter: no OPENROUTER_API_KEY found in your environment / .env file.
[OK] Ollama (gemma4:e2b-mlx): Hello! How can I help you today?

You only need ONE of these to work.


## Step 4: Choose Your Default API Provider

If your APIs are working, you can set a default for the remaining lessons.
All lesson notebooks support OpenAI, DeepSeek, OpenRouter, and Ollama (local) — switch with the `PROVIDER` line at the top of each notebook.

In [4]:
print("API Configuration Status:")
for label, cfg in PROVIDERS.items():
    if cfg['env'] is None:
        print(f"  {label + ':':<12} {cfg['base_url']} (make sure `ollama serve` is running)")
    else:
        state = 'Configured' if os.getenv(cfg['env']) else f"Not set ({cfg['env']})"
        print(f"  {label + ':':<12} {state}")

print("\nIn every lesson notebook, pick your provider by editing ONE line:")
print("    PROVIDER = 'openai'   # or 'deepseek' / 'openrouter' / 'ollama'")
print("\nYou're all set! Start with Lesson 01: 走进AI工程化 / Introduction to AI Engineering.")


API Configuration Status:
  OpenAI:      Not set (OPENAI_API_KEY)
  DeepSeek:    Configured
  OpenRouter:  Not set (OPENROUTER_API_KEY)
  Ollama:      http://localhost:11434/v1 (make sure `ollama serve` is running)

In every lesson notebook, pick your provider by editing ONE line:
    PROVIDER = 'openai'   # or 'deepseek' / 'openrouter' / 'ollama'

You're all set! Start with Lesson 01: 走进AI工程化 / Introduction to AI Engineering.
